In [1]:
import numpy as np


class MultinomialNaiveBayesScratch:

  def __init__(self, alpha=1.0):
    self.alpha = alpha  # Laplace smoothing parameter
    self.classes = None
    self.class_priors = {}
    self.feature_probs = {}

  def fit(self, X, y):
    """X: Feature matrix of shape (m, n) containing word count vectors

    y: Target class labels vector of shape (m,)
    """
    m, n = X.shape
    self.classes = np.unique(y)

    for c in self.classes:
      # Filter samples belonging to class c
      X_c = X[y == c]

      # 1. Calculate Prior P(Y = c)
      self.class_priors[c] = X_c.shape[0] / m

      # 2. Calculate Likelihood P(x_i | Y = c) with Laplace Smoothing
      # Sum of counts for feature i across all class c samples
      feature_counts = np.sum(X_c, axis=0)
      # Total word counts in class c
      total_count = np.sum(feature_counts)

      # P(x_i | Y) = (Count(x_i, c) + alpha) / (Total_Count(c) + alpha * Vocabulary_Size)
      probs = (feature_counts + self.alpha) / (
          total_count + self.alpha * n
      )
      self.feature_probs[c] = probs

  def predict_log_proba(self, X):
    """Predicts log posterior probabilities to avoid floating point underflow."""
    m, n = X.shape
    log_posteriors = np.zeros((m, len(self.classes)))

    for idx, c in enumerate(self.classes):
      # log P(Y) + sum(x_i * log P(x_i | Y))
      prior_log = np.log(self.class_priors[c])
      likelihood_log = np.dot(X, np.log(self.feature_probs[c]))
      log_posteriors[:, idx] = prior_log + likelihood_log

    return log_posteriors

  def predict(self, X):
    log_posteriors = self.predict_log_proba(X)
    class_indices = np.argmax(log_posteriors, axis=1)
    return self.classes[class_indices]


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == '__main__':
  # Dummy Bag-of-Words Vocabulary: ["offer", "free", "learning", "python", "money"]
  # X: Word counts in document samples
  X_train = np.array([
      [3, 2, 0, 0, 1],  # Spam sample 1
      [2, 3, 0, 0, 2],  # Spam sample 2
      [0, 0, 3, 2, 0],  # Ham sample 1
      [0, 1, 2, 4, 0],  # Ham sample 2
  ])
  y_train = np.array(['Spam', 'Spam', 'Ham', 'Ham'])

  model = MultinomialNaiveBayesScratch(alpha=1.0)
  model.fit(X_train, y_train)

  # Test document: "free python offer" -> [1, 1, 0, 1, 0]
  X_test = np.array([[1, 1, 0, 1, 0]])
  prediction = model.predict(X_test)[0]

  print('=' * 55)
  print('  NAIVE BAYES TEXT CLASSIFICATION DEMO')
  print('=' * 55)
  print(f'Test Document Features: {X_test[0]}')
  print(f'Predicted Category   : {prediction}')
  print('=' * 55)

  NAIVE BAYES TEXT CLASSIFICATION DEMO
Test Document Features: [1 1 0 1 0]
Predicted Category   : Spam


In [ ]:
#When adding your own custom dataset to Naive Bayes, there are three critical steps to prepare your raw text data before feeding it into the model
# 1. The Raw Text Preprocessing Pipeline
# 2. Building a Bag-of-Words (BoW) Vectorizer
# 3. Complete End-to-End Execution with Custom Data

In [3]:
import re
from collections import Counter
import numpy as np


class SimpleCountVectorizer:

  def __init__(self, max_features=1000):
    self.max_features = max_features
    self.vocabulary_ = {}

  def preprocess(self, text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|[^\w\s]', '', text)
    return text.strip()

  def fit_transform(self, raw_documents):
    all_words = []
    processed_docs = [self.preprocess(doc).split() for doc in raw_documents]

    for words in processed_docs:
      all_words.extend(words)

    most_common = Counter(all_words).most_common(self.max_features)
    self.vocabulary_ = {word: i for i, (word, _) in enumerate(most_common)}

    X = np.zeros((len(raw_documents), len(self.vocabulary_)), dtype=int)
    for row_idx, words in enumerate(processed_docs):
      for word in words:
        if word in self.vocabulary_:
          X[row_idx, self.vocabulary_[word]] += 1
    return X

  def transform(self, raw_documents):
    X = np.zeros((len(raw_documents), len(self.vocabulary_)), dtype=int)
    for row_idx, doc in enumerate(raw_documents):
      words = self.preprocess(doc).split()
      for word in words:
        if word in self.vocabulary_:
          X[row_idx, self.vocabulary_[word]] += 1
    return X

import pandas as pd

# Load your custom dataset (CSV or List)
# df = pd.read_csv('your_dataset.csv')
# texts = df['text_column'].values
# labels = df['label_column'].values

texts = [
    'Win a free cash prize now click link',
    'Important project update for tomorrow meeting',
    'Earn money fast with zero risk guarantee',
    'Can we schedule the code review session today',
]
labels = ['Spam', 'Ham', 'Spam', 'Ham']

# Vectorize Text
vectorizer = SimpleCountVectorizer(max_features=500)
X_train = vectorizer.fit_transform(texts)
y_train = np.array(labels)

# Train Naive Bayes Model
model = MultinomialNaiveBayesScratch(alpha=1.0)
model.fit(X_train, y_train)

# Predict on Unseen Sentences
new_samples = ['Free cash offer inside', 'Meeting scheduled for tomorrow']
X_test = vectorizer.transform(new_samples)
predictions = model.predict(X_test)

for sample, pred in zip(new_samples, predictions):
  print(f"📩 Text: '{sample}' ---> Category: [{pred}]")

📩 Text: 'Free cash offer inside' ---> Category: [Spam]
📩 Text: 'Meeting scheduled for tomorrow' ---> Category: [Ham]
